<a href="https://colab.research.google.com/github/rahulkeshamoni129/-Development-of-Interactive-Cyber-Threat-Visualization-Dashboard-/blob/main/SQL_Cyberthreat_infosys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SQL

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

Data Generation Prompt

In [ ]:
np.random.seed(42)

rows = 1500

cyber_data = pd.DataFrame({
    "incident_id": range(1, rows + 1),
    "attack_type": np.random.choice(
        [
            "Phishing", "Malware", "DDoS", "Ransomware",
            "SQL Injection", "Brute Force", "XSS", "Trojan"
        ],
        rows
    ),
    "severity": np.random.choice(
        ["Low", "Medium", "High", "Critical"],
        rows,
        p=[0.35, 0.30, 0.25, 0.10]
    ),
    "system_id": np.random.randint(101, 121, rows),
    "country": np.random.choice(
        [
            "India", "USA", "UK", "Germany", "France",
            "China", "Russia", "Brazil", "Canada", "Australia"
        ],
        rows
    ),
    "date": pd.to_datetime("2023-01-01") +
            pd.to_timedelta(
                np.random.randint(0, 600, rows),
                unit="D"
            )
})

cyber_data.to_csv("cyber_threat_dataset.csv", index=False)

cyber_data.head()


,incident_id,attack_type,severity,system_id,country,date
0,1,XSS,Low,109,USA,2023-10-18
1,2,Ransomware,Critical,109,Germany,2024-04-29
2,3,SQL Injection,Medium,104,USA,2023-11-02
3,4,XSS,Low,103,India,2023-05-08
4,5,DDoS,Low,117,Russia,2023-09-30


Context Prompt

In [ ]:
systems = pd.DataFrame({
    "system_id": range(101, 121),
    "system_name": [
        "Web Server", "Database", "Email Server", "Firewall", "User PC",
        "Cloud VM", "API Gateway", "Load Balancer", "Proxy", "Auth Server",
        "Backup Server", "DNS", "SIEM", "IDS", "IPS", "VPN", "ERP", "CRM",
        "Dev Server", "Test Server"
    ]
})

systems.head()


,system_id,system_name
0,101,Web Server
1,102,Database
2,103,Email Server
3,104,Firewall
4,105,User PC


Database Prompt

In [ ]:
conn = sqlite3.connect("cyber_threat.db")

cyber_data.to_sql(
    "incidents",
    conn,
    index=False,
    if_exists="replace"
)

systems.to_sql(
    "systems",
    conn,
    index=False,
    if_exists="replace"
)


20

SELECT Prompt

In [ ]:
# View first 10 records
pd.read_sql(
    "SELECT * FROM incidents LIMIT 10",
    conn
)


,incident_id,attack_type,severity,system_id,country,date
0,1,XSS,Low,109,USA,2023-10-18 00:00:00
1,2,Ransomware,Critical,109,Germany,2024-04-29 00:00:00
2,3,SQL Injection,Medium,104,USA,2023-11-02 00:00:00
3,4,XSS,Low,103,India,2023-05-08 00:00:00
4,5,DDoS,Low,117,Russia,2023-09-30 00:00:00
5,6,Trojan,High,102,Brazil,2024-03-17 00:00:00
6,7,SQL Injection,Medium,101,China,2024-05-17 00:00:00
7,8,SQL Injection,Medium,108,Germany,2023-05-10 00:00:00
8,9,XSS,High,113,China,2023-05-12 00:00:00
9,10,Malware,High,115,UK,2024-08-19 00:00:00


WHERE Prompt

In [ ]:
# Filter High and Critical attacks
pd.read_sql(
    "SELECT * FROM incidents WHERE severity IN ('High','Critical')",
    conn
)


,incident_id,attack_type,severity,system_id,country,date
0,2,Ransomware,Critical,109,Germany,2024-04-29 00:00:00
1,6,Trojan,High,102,Brazil,2024-03-17 00:00:00
2,9,XSS,High,113,China,2023-05-12 00:00:00
3,10,Malware,High,115,UK,2024-08-19 00:00:00
4,14,DDoS,High,114,France,2023-11-21 00:00:00
...,...,...,...,...,...,...
541,1487,XSS,High,109,Russia,2023-08-03 00:00:00
542,1491,Phishing,High,107,China,2024-06-24 00:00:00
543,1494,Malware,Critical,115,France,2023-11-30 00:00:00
544,1495,Brute Force,High,108,Canada,2023-10-31 00:00:00


GROUP BY Prompt

In [ ]:
# Count attacks per type
pd.read_sql(
    "SELECT attack_type, COUNT(*) AS total FROM incidents GROUP BY attack_type",
    conn
)


,attack_type,total
0,Brute Force,186
1,DDoS,176
2,Malware,180
3,Phishing,203
4,Ransomware,193
5,SQL Injection,191
6,Trojan,188
7,XSS,183


ORDER BY Prompt

In [ ]:
# Order by most recent attacks
pd.read_sql(
    "SELECT * FROM incidents ORDER BY date DESC LIMIT 10",
    conn
)


,incident_id,attack_type,severity,system_id,country,date
0,908,DDoS,Low,111,China,2024-08-22 00:00:00
1,49,Brute Force,High,103,Russia,2024-08-21 00:00:00
2,173,DDoS,High,115,India,2024-08-21 00:00:00
3,765,SQL Injection,Low,109,India,2024-08-21 00:00:00
4,1155,Trojan,High,116,China,2024-08-21 00:00:00
5,1166,Malware,High,103,UK,2024-08-21 00:00:00
6,1500,XSS,Low,114,China,2024-08-21 00:00:00
7,10,Malware,High,115,UK,2024-08-19 00:00:00
8,1256,Malware,High,116,UK,2024-08-19 00:00:00
9,1379,Brute Force,High,106,Canada,2024-08-19 00:00:00


INNER JOIN Prompt

In [ ]:
# Match incidents with systems
pd.read_sql(
    "SELECT i.incident_id, i.attack_type, s.system_name "
    "FROM incidents i INNER JOIN systems s "
    "ON i.system_id = s.system_id",
    conn
)


,incident_id,attack_type,system_name
0,1,XSS,Proxy
1,2,Ransomware,Proxy
2,3,SQL Injection,Firewall
3,4,XSS,Email Server
4,5,DDoS,ERP
...,...,...,...
1495,1496,SQL Injection,DNS
1496,1497,Trojan,CRM
1497,1498,Trojan,IPS
1498,1499,DDoS,ERP


LEFT JOIN Prompt

In [ ]:
# Include all incidents even without matching system
pd.read_sql(
    "SELECT i.incident_id, i.attack_type, s.system_name "
    "FROM incidents i LEFT JOIN systems s "
    "ON i.system_id = s.system_id",
    conn
)


,incident_id,attack_type,system_name
0,1,XSS,Proxy
1,2,Ransomware,Proxy
2,3,SQL Injection,Firewall
3,4,XSS,Email Server
4,5,DDoS,ERP
...,...,...,...
1495,1496,SQL Injection,DNS
1496,1497,Trojan,CRM
1497,1498,Trojan,IPS
1498,1499,DDoS,ERP


SUBQUERY Prompt

In [ ]:
# Find systems attacked more than 50 times
pd.read_sql(
    "SELECT * FROM incidents WHERE system_id IN ("
    "SELECT system_id FROM incidents GROUP BY system_id HAVING COUNT(*) > 50"
    ")",
    conn
)


,incident_id,attack_type,severity,system_id,country,date
0,1,XSS,Low,109,USA,2023-10-18 00:00:00
1,2,Ransomware,Critical,109,Germany,2024-04-29 00:00:00
2,3,SQL Injection,Medium,104,USA,2023-11-02 00:00:00
3,4,XSS,Low,103,India,2023-05-08 00:00:00
4,5,DDoS,Low,117,Russia,2023-09-30 00:00:00
...,...,...,...,...,...,...
1495,1496,SQL Injection,Medium,112,Brazil,2024-03-12 00:00:00
1496,1497,Trojan,Low,118,Germany,2023-11-19 00:00:00
1497,1498,Trojan,High,115,France,2024-05-16 00:00:00
1498,1499,DDoS,Medium,117,France,2024-01-28 00:00:00


AGGREGATION Prompt

In [ ]:
# Country-wise aggregation
pd.read_sql(
    "SELECT country, COUNT(*) AS attacks FROM incidents GROUP BY country",
    conn
)


,country,attacks
0,Australia,151
1,Brazil,157
2,Canada,146
3,China,169
4,France,127
5,Germany,157
6,India,134
7,Russia,169
8,UK,149
9,USA,141


Visualization Prompt


Count the total number of incidents for each severity level and visualize the distribution using a bar chart.


In [ ]:
severity_counts = pd.read_sql(
    "SELECT severity, COUNT(*) AS total_incidents FROM incidents GROUP BY severity ORDER BY total_incidents DESC",
    conn
)

print("Incident counts by severity level:")
print(severity_counts)

Incident counts by severity level:
   severity  total_incidents
0       Low              524
1    Medium              430
2      High              403
3  Critical              143



Identify the top 5 `system_name` values that have experienced the most incidents and visualize them using a bar chart.


In [ ]:
top_5_systems = pd.read_sql(
    "SELECT s.system_name, COUNT(i.incident_id) AS incident_count "
    "FROM incidents i "
    "INNER JOIN systems s ON i.system_id = s.system_id "
    "GROUP BY s.system_name "
    "ORDER BY incident_count DESC "
    "LIMIT 5",
    conn
)

print("Top 5 most attacked systems:")
print(top_5_systems)

Top 5 most attacked systems:
  system_name  incident_count
0         ERP              99
1        SIEM              90
2         IPS              88
3  Dev Server              85
4    Firewall              82



Analyze the count of each `attack_type` for the top 3 countries with the most incidents and visualize this distribution.


In [ ]:
top_3_countries = pd.read_sql(
    "SELECT country, COUNT(*) AS total_incidents FROM incidents GROUP BY country ORDER BY total_incidents DESC LIMIT 3",
    conn
)

print("Top 3 countries with the most incidents:")
print(top_3_countries)

Top 3 countries with the most incidents:
   country  total_incidents
0   Russia              169
1    China              169
2  Germany              157


In [ ]:
top_countries_list = top_3_countries['country'].tolist()

attack_type_by_country = pd.read_sql(
    f"SELECT country, attack_type, COUNT(*) AS total_attacks "
    f"FROM incidents "
    f"WHERE country IN ({str(top_countries_list)[1:-1]}) "
    f"GROUP BY country, attack_type "
    f"ORDER BY country, total_attacks DESC",
    conn
)

# Pivot the table for easier plotting
pivoted_attack_data = attack_type_by_country.pivot(index='attack_type', columns='country', values='total_attacks').fillna(0)

print("Attack type distribution for top 3 countries:")
print(pivoted_attack_data)

Attack type distribution for top 3 countries:
country        China  Germany  Russia
attack_type                          
Brute Force       18       18      26
DDoS              16       15      20
Malware           17       17      19
Phishing          26       18      21
Ransomware        25       23      17
SQL Injection     28       23      25
Trojan            20       19      20
XSS               19       24      21



Calculate and visualize the monthly incident count broken down by `severity` using a line plot to show trends over time.


In [ ]:
monthly_severity_counts = pd.read_sql(
    "SELECT STRFTIME('%Y-%m', date) AS month_year, severity, COUNT(*) AS incident_count "
    "FROM incidents "
    "GROUP BY month_year, severity "
    "ORDER BY month_year, severity",
    conn
)

print("Monthly incident counts by severity:")
print(monthly_severity_counts.head())

Monthly incident counts by severity:
  month_year  severity  incident_count
0    2023-01  Critical               5
1    2023-01      High              14
2    2023-01       Low              27
3    2023-01    Medium              24
4    2023-02  Critical               8
